# Collecte des données (suite) #
>
>## III - Données météorologiques ##
>>
>>Nous avons précédemment collecté les données de **production d'énergie solaire** et les données sur la **position du soleil**. Or, parmi les connaissances de l'homme du métier, d'autres données que celles concernant la position du soleil pourraient permettre d'expliquer notre variable cible et être utiles à notre modélisation.
>>
>>Ainsi, dans ce notebook, nous nous intéressons aux *rayonnements solaires effectivements reçus* au niveau des panneaux solaires et donc à des données concernant l'**athmosphère**.
>>
>>Ces données sont par exemple disponibles dans un jeu de données `CAMS solar radiation time-serie` mis gracieusement à disposition par Copernicus (programme
d’observation de la Terre de l’Union européenne - https://www.copernicus.eu/fr) via son service de surveillance de l’atmosphère (CAMS - https://atmosphere.copernicus.eu/).
>>
>Pour chacun des points d'intérêts identifiés pour la production d'énergie solaire, nous récupèrerons les variables :
>- **GHI** : irradiance horizontale globale *(mesure globale de la lumière solaire reçue sur un plan horizontal au niveau du sol)* ;
>- **BHI** : irradiance horizontale directe *(mesure des rayonnements solaires directs reçus sur un plan horizontal au niveau du sol)* ;
>- **DHI** : irradiance horizontale diffuse *(mesure des rayonnements solaires indirects, diffusés dans l'athmosphère reçus sur un plan horizontal au niveau du sol)* ;
>- **BNI** : irradiance normale directe *(mesure des rayonnements solaires directs reçus sur un plan perpendiculaire aux rayons solaires et se déplaçant avec ce dernier)*.
>
>Pour chacune de ces variables, nous récupèreront également leur valeur théorique maximale (pour un ciel dégagé).
>


>> ### A - Récupération des données de production et des données astronomiques ###
>>>
>>>Nous commencons par récupérer les données aggrégées à l'étape précédente depuis le Drive du projet :


In [1]:
# On se donne accès au Drive qui contient les fichiers sources
from google.colab import drive

# Monter Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Chemin pour récupérer le dataset de donnees astronomiques
chemin_dataset_astro = '/content/drive/MyDrive/Flux/00_Collecte_datasets/02_Astronomie/' + "astro_2020_2024.csv"

# Chemin vers le répertoire de travail pour les datasets de données météo
chemin_datasets_meteo = '/content/drive/MyDrive/Flux/00_Collecte_datasets/03_Athmosphere/'

>>>On importe les librairies nécessaires pour manipuler nos données :

In [3]:
# On importe les librairies dont on a besoin pour traiter les datasets
import pandas as pd
import numpy as np

>>>On charge le jeu de données de l'étape précédente :

In [4]:
# On récupère le dataset contenant les données de production et les données astronomiques
df_astro = pd.read_csv(chemin_dataset_astro, parse_dates=['datetime_utc'])
display(df_astro.head())
df_astro.dtypes

/tmp/ipython-input-3255259691.py:2: DtypeWarning: Columns (6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_astro = pd.read_csv(chemin_dataset_astro, parse_dates=['datetime_utc'])


,datetime_utc,Périmètre,Nature,Consommation,Solaire,Ech. physiques,Stockage batterie,Déstockage batterie,TCO Solaire (%),TCH Solaire (%),CRU_azimuth,CRU_altitude,SEL_azimuth,SEL_altitude,SVT_azimuth,SVT_altitude,BRA_azimuth,BRA_altitude,EYG_azimuth,EYG_altitude
0,2019-12-31 23:00:00+00:00,PACA,Données définitives,6123.0,0.0,3332.0,-,-,0.0,0.0,335.596867,-67.454606,336.734433,-67.160776,337.561520,-68.097845,335.241040,-68.046190,333.265411,-67.469007
1,2019-12-31 23:30:00+00:00,PACA,Données définitives,5907.0,0.0,2837.0,-,-,0.0,0.0,353.817769,-68.883102,354.738150,-68.485180,356.336005,-69.329403,353.945722,-69.500739,351.451534,-69.119526
2,2020-01-01 00:00:00+00:00,PACA,Données définitives,5724.0,0.0,2668.0,-,-,0.0,0.0,12.883689,-68.564280,13.430816,-68.099948,15.637215,-68.757908,13.536657,-69.141123,10.834111,-69.009045
3,2020-01-01 00:30:00+00:00,PACA,Données définitives,5749.0,0.0,2754.0,-,-,0.0,0.0,30.261125,-66.570892,30.451783,-66.089459,32.859299,-66.517353,31.240402,-67.054165,28.719513,-67.163582
4,2020-01-01 01:00:00+00:00,PACA,Données définitives,5700.0,0.0,2886.0,-,-,0.0,0.0,44.631373,-63.284229,44.594428,-62.821307,46.883423,-63.030485,45.699703,-63.656435,43.545408,-63.957709


,0
datetime_utc,"datetime64[ns, UTC]"
Périmètre,object
Nature,object
Consommation,float64
Solaire,float64
Ech. physiques,float64
Stockage batterie,object
Déstockage batterie,object
TCO Solaire (%),float64
TCH Solaire (%),float64


>>### B - Lieux retenus ###
>>>
>>>On récupère les coordonnées des cinq points significatifs que l'on a précédemment déterminé.
>>>
>>>Pour faciliter l'exploitation ultérieure des données géographiques, on remplace le nom de villes sélectionnées par un code sur trois lettres :    
>>>
>>> - Cruis est encodé par 'CRU' ;
>>> - Saint-Étienne-le-Laus par 'SEL' ;
>>> - Saint-Vallier-de-Thiey par 'SVT' ;
>>> - Bras par 'BRA' ;
>>> - Eygalières par 'EYG'.

In [5]:
# On a retenu 5 lieux pertinents pour l'étude de production d'énergie solaire
# On stocke leur code ainsi que leur coordonnées géographiques (latitude et longitude) dans un dictionnaire
lieux_retenus = {'CRU' : {'latitude': 44.0845, 'longitude': 5.8397},
                 'SEL' : {'latitude': 44.5075, 'longitude': 6.1616},
                 'SVT' : {'latitude': 43.6994, 'longitude': 6.8516},
                 'BRA' : {'latitude': 43.4723, 'longitude': 5.9558},
                 'EYG' : {'latitude': 43.7638, 'longitude': 4.9554}}

>>## C - Collecte des données athmosphériques ##
>>>
>>>Le jeu de données **CAMS solar radiation time-serie** (https://ads.atmosphere.copernicus.eu/datasets/cams-solar-radiation-timeseries?tab=overview) est accessible via une API par l'intermédiaire de la librairie cdsapi.



>>>On commence par installer la librairie qui va nous permettre d'accéder à l'API du jeu de données :

In [6]:
# On installe la librairie dont on a besoin pour se connecter à l'API
!pip install cdsapi

>>>Puis on l'importe avec d'autres librairies utiles pour la collecte :

In [7]:
# On importe les librairies dont on a besoin pour accéder aux données
import yaml
import os
import cdsapi
import time

>>>L'accès à l'API se fait via un compte : on charge donc les données correspondantes pour accéder à l'API :

In [8]:
# Répertoire utilisateur
home_dir = os.path.expanduser("~")

# URL de l'API
url = "https://ads.atmosphere.copernicus.eu/api"

# API Token
key = "6f9b0086-c5eb-4122-a89e-8dc8c99da391" # groupe
#key = "47759f34-4527-4bb8-9d97-6406d3354b21" # Christophe
credentials = {"url": url, "key": key}

# Création des fichiers dont on a besoin pour se connecter à l'API
# On écrit le fichier ".ecmwfdatastoresrc"
with open(f"{home_dir}/.ecmwfdatastoresrc", "w") as f:
    yaml.safe_dump(credentials, f)

# Emplacement attendu par cdsapi:
cdsapirc_path = os.path.expanduser("~/.cdsapirc")

# On écrit le fichier ".cdsapirc"
with open(cdsapirc_path, "w") as f:
    f.write(f"url: {url}\n")
    f.write(f"key: {key}\n")


>>>Pour faciliter la collecte des données pour chacun de nos points d'intérêts, on crée une fonction qui interroge l'API en fonction des coordonnées géographiques d'un lieu (latitude et longitude) et des dates de début et de fin de la période à observer.
>>>
>>>Un jeu de données est alors enregistré au niveau d'un chemin transmis à la fonction.

In [9]:
def retrieve_ghi (latitude, longitude, date_debut, date_fin, base_chemin) :
  # On crée le client de l'API
  client = cdsapi.Client()

  # Nom du jeu de données qui nous intéresse
  dataset = "cams-solar-radiation-timeseries"

  # Formulation de la requête
  requete = {"sky_type": "observed_cloud",
             "location": {"longitude": longitude, "latitude": latitude},
             "altitude": ["0"],
             "date": [date_debut + "/" + date_fin],
             "time_step": "15minute", # '30minute' n'existe pas on passe directement à un intervalle d'une heure
             "time_reference": "universal_time",
             "data_format": "csv"}

  target = base_chemin + "_" +date_debut + "_" + date_fin + ".csv"

  # Envoi de la requête
  fname = client.retrieve(dataset, requete, target)

>>>On détermine, à partir du jeu de données précédemment collecté, la date de début et la date de fin de la période couverte.

In [10]:
# On détermine la plage de dates à requêter
# En heures UTC le dataset de départ commence le 31 décembre 2019 et se termine le 31 décembre 2024

date_debut = df_astro.loc[0,'datetime_utc'].strftime('%Y-%m-%d')
date_fin = df_astro.loc[df_astro.shape[0]-1,'datetime_utc'].strftime('%Y-%m-%d')

print("Date de début :", date_debut)
print("Date de fin :", date_fin)


Date de début : 2019-12-31
Date de fin : 2024-12-31


>>>On interroge l'API pour collecter les données athmosphérique pour chaque point d'intérêt :

In [11]:
# On interroge l'API pour récupérer les données CAMS de chaque ville
# Pour chaque ville
for ville, location in lieux_retenus.items() :
  # On affiche où on en est rendu dans les villes
  print(ville + "...")

  # Le chemin de base du dataset temporaire sera :
  base_chemin = chemin_datasets_meteo + "CAMS_" + ville # La fonction de récupération ajoutera les dates et l'extension

  # Envoi de la requete
  retrieve_ghi(location['latitude'], location['longitude'],
               date_debut, date_fin,
               base_chemin)

  # On indique que la requete est terminée
  print("Ok pour", ville)

  # On patiente avant la requête suivante
  time.sleep(10)


CRU...


2026-01-16 13:17:05,394 INFO Request ID is 1d638755-f3f4-492e-93d1-17b06e921c6b
INFO:ecmwf.datastores.legacy_client:Request ID is 1d638755-f3f4-492e-93d1-17b06e921c6b
2026-01-16 13:17:05,544 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-01-16 13:17:19,450 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-01-16 13:18:21,696 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


e2ca8bd5ae95eb6d303fce5be96088c7.csv:   0%|          | 0.00/20.1M [00:00<?, ?B/s]

Ok pour CRU
SEL...


2026-01-16 13:18:35,232 INFO Request ID is bd9e6ae4-ae97-4bb0-953c-1ae920e3b365
INFO:ecmwf.datastores.legacy_client:Request ID is bd9e6ae4-ae97-4bb0-953c-1ae920e3b365
2026-01-16 13:18:35,392 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-01-16 13:18:49,301 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-01-16 13:19:25,900 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


...

1c6257d95e94a326368b64bf86dc76a8.csv:   0%|          | 0.00/20.1M [00:00<?, ?B/s]

Ok pour BRA
EYG...


2026-01-16 13:21:50,548 INFO Request ID is 24720b6c-fa03-4fff-82aa-ad9d8f62e1d4
INFO:ecmwf.datastores.legacy_client:Request ID is 24720b6c-fa03-4fff-82aa-ad9d8f62e1d4
2026-01-16 13:21:50,708 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-01-16 13:22:04,633 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-01-16 13:23:06,942 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


4a71b35047777f39cfb428777666b8be.csv:   0%|          | 0.00/20.1M [00:00<?, ?B/s]

Ok pour EYG


>>### D - Aggrégation des nouvelles données collectées ###
>>>
>>>Les données collectées se présentent pour le moment sous la forme de fichiers csv enregistrés dans le Drive du projet, à raison d'un fichier par point d'intérêt.
>>>
>>>Pour aggréger les nouvelles variables explicatives, on va :
>>>
>>>- Charger les fichiers sous forme de DataFrame Pandas ;
>>>- Renommer les variables en les faisant commencer par le sigle du point d'intérêt ;
>>>-Fusionner les jeux de données sur la base de la variable temporelle.


>>>On charge sous forme de DataFrame Pandas les jeux de données qu'on vient de collecter :

In [12]:
# On crée une fonction lambda pour obtenir la date au bon format
# (on conserve la borne de début de l'intervalle de temps)
f = lambda x: pd.to_datetime(x.split('/')[0]).tz_localize("UTC")

In [14]:
# On charge les dataset sous forme de DataFrame pour pouvoir les traiter
# Attention : environ 10 minutes d'exécution...
print("Cruis...")
cams_CRU = pd.read_csv(chemin_datasets_meteo + 'CAMS_CRU_2019-12-31_2024-12-31.csv',
                      sep=';', header=42,
                      converters={'# Observation period': f})

print('Saint-Étienne-le-Laus')
cams_SEL = pd.read_csv(chemin_datasets_meteo + 'CAMS_SEL_2019-12-31_2024-12-31.csv',
                      sep=';', header=42,
                      converters={'# Observation period': f})

print('Saint-Vallier-de-Thiey')
cams_SVT = pd.read_csv(chemin_datasets_meteo + 'CAMS_SVT_2019-12-31_2024-12-31.csv',
                      sep=';', header=42,
                      converters={'# Observation period': f})

print('Bras')
cams_BRA = pd.read_csv(chemin_datasets_meteo + 'CAMS_BRA_2019-12-31_2024-12-31.csv',
                      sep=';', header=42,
                      converters={'# Observation period': f})
print("Eygalières...")
cams_EYG = pd.read_csv(chemin_datasets_meteo + 'CAMS_EYG_2019-12-31_2024-12-31.csv',
                      sep=';', header=42,
                      converters={'# Observation period': f})

Cruis...
Saint-Étienne-le-Laus
Saint-Vallier-de-Thiey
Bras
Eygalières...


In [ ]:
#df_cams_30 = (
#    df.groupby("commune")
#    .resample("30T")
#    .agg({**{c: "sum" for c in energy_cols},  # somme des énergies 15 min -> 30 min})
#    .reset_index())

# Colonnes d'énergie à sommer
#energy_cols = ["toa", "ghi_cs", "bhi_cs", "dhi_cs", "bni_cs", "ghi", "bhi", "dhi", "bni"]

>>>On réalise une première observation des jeux de données collectés :

In [15]:
# On commence par regarder à quoi ressemble un dataset cams
cams_CRU.info()
display(cams_CRU.describe())
cams_CRU.nunique()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175488 entries, 0 to 175487
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype              
---  ------                --------------   -----              
 0   # Observation period  175488 non-null  datetime64[ns, UTC]
 1   TOA                   175488 non-null  float64            
 2   Clear sky GHI         175488 non-null  float64            
 3   Clear sky BHI         175488 non-null  float64            
 4   Clear sky DHI         175488 non-null  float64            
 5   Clear sky BNI         175488 non-null  float64            
 6   GHI                   175488 non-null  float64            
 7   BHI                   175488 non-null  float64            
 8   DHI                   175488 non-null  float64            
 9   BNI                   175488 non-null  float64            
 10  Reliability           175488 non-null  float64            
dtypes: datetime64[ns, UTC](1), float64(10)
memory usage:

,TOA,Clear sky GHI,Clear sky BHI,Clear sky DHI,Clear sky BNI,GHI,BHI,DHI,BNI,Reliability
count,175488.000000,175488.000000,175488.000000,175488.000000,175488.000000,175488.000000,175488.000000,175488.000000,175488.000000,175488.000000
mean,77.755296,56.941334,45.531930,11.409404,85.715740,45.993977,30.537864,15.456113,57.820092,0.978015
std,98.561004,75.791909,63.332389,14.703329,96.251817,67.071796,53.286388,23.586732,83.829046,0.094220
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
50%,2.681750,0.703850,0.063350,0.612600,1.248400,0.494900,0.000000,0.451300,0.000000,1.000000
75%,150.655975,108.472650,85.880775,20.895650,192.007125,81.157225,42.042975,23.361600,123.476800,1.000000
max,308.452000,253.079600,232.761400,112.624400,258.484300,253.079600,232.761400,134.651200,258.484300,1.000000


,0
# Observation period,175488
TOA,88199
Clear sky GHI,87601
Clear sky BHI,85836
Clear sky DHI,80646
Clear sky BNI,85676
GHI,87237
BHI,74952
DHI,83318
BNI,77922


>>>Pour parcourir plus facilement les datasets on utilise un dictionnaires ayant pour clé le sigle correspondant au point d'intérêt du dataset :

In [67]:
# 'Cruis' 'Saint-Étienne-le-Laus' 'Saint-Vallier-de-Thiey'  'Bras'  'Eygalières'
all_cams = {'CRU' : cams_CRU,
            'SEL' : cams_SEL,
            'SVT' : cams_SVT,
            'BRA' : cams_BRA,
            'EYG' : cams_EYG}

# On affiche quelques valeurs vers 10h du matin
for df in all_cams.values():
  display(df.iloc[42:51, :])
  #display(df.head())

,# Observation period,TOA,Clear sky GHI,Clear sky BHI,Clear sky DHI,Clear sky BNI,GHI,BHI,DHI,BNI,Reliability
42,2019-12-31 10:30:00+00:00,127.9681,93.9969,79.7676,14.2293,219.3695,93.9969,79.7676,14.2293,219.3695,1.0
43,2019-12-31 10:45:00+00:00,131.5526,97.1360,82.7308,14.4052,221.3223,97.1360,82.7308,14.4052,221.3223,1.0
44,2019-12-31 11:00:00+00:00,134.1630,99.4364,84.9078,14.5286,222.7285,99.4364,84.9078,14.5286,222.7285,1.0
45,2019-12-31 11:15:00+00:00,135.7882,100.8832,86.2814,14.6018,223.6242,100.8832,86.2814,14.6018,223.6242,1.0
46,2019-12-31 11:30:00+00:00,136.4211,101.4670,86.8410,14.6260,224.0310,101.4670,86.8410,14.6260,224.0310,1.0
47,2019-12-31 11:45:00+00:00,136.0591,101.1840,86.5821,14.6019,223.9572,101.1840,86.5821,14.6019,223.9572,1.0
48,2019-12-31 12:00:00+00:00,134.7037,100.0313,85.5051,14.5263,223.3960,100.0313,85.5051,14.5263,223.3960,1.0
49,2019-12-31 12:15:00+00:00,132.3607,98.0161,83.6177,14.3984,222.3308,98.0161,83.6177,14.3984,222.3308,1.0
50,2019-12-31 12:30:00+00:00,129.0402,95.1549,80.9356,14.2193,220.7347,95.1549,80.9356,14.2193,220.7347,1.0


,# Observation period,TOA,Clear sky GHI,Clear sky BHI,Clear sky DHI,Clear sky BNI,GHI,BHI,DHI,BNI,Reliability
42,2019-12-31 10:30:00+00:00,125.9731,93.6346,80.5471,13.0875,225.0217,93.6346,80.5471,13.0875,225.0217,1.0
43,2019-12-31 10:45:00+00:00,129.4494,96.6949,83.4551,13.2398,226.8875,96.6949,83.4551,13.2398,226.8875,1.0
44,2019-12-31 11:00:00+00:00,131.9575,98.9171,85.5725,13.3446,228.2243,98.9171,85.5725,13.3446,228.2243,1.0
45,2019-12-31 11:15:00+00:00,133.4866,100.2871,86.8832,13.4038,229.0667,100.2871,86.8832,13.4038,229.0667,1.0
46,2019-12-31 11:30:00+00:00,134.0303,100.7959,87.3771,13.4187,229.4350,100.7959,87.3771,13.4187,229.4350,1.0
47,2019-12-31 11:45:00+00:00,133.5861,100.4401,87.0503,13.3898,229.3367,100.4401,87.0503,13.3898,229.3367,1.0
48,2019-12-31 12:00:00+00:00,132.1560,99.2158,85.8942,13.3216,228.7387,99.2158,85.8942,13.3216,228.7387,1.0
49,2019-12-31 12:15:00+00:00,129.7461,97.1301,83.9170,13.2131,227.6228,97.1301,83.9170,13.2131,227.6228,1.0
50,2019-12-31 12:30:00+00:00,126.3667,94.2034,81.1454,13.0580,225.9886,94.2034,81.1454,13.0580,225.9886,1.0


,# Observation period,TOA,Clear sky GHI,Clear sky BHI,Clear sky DHI,Clear sky BNI,GHI,BHI,DHI,BNI,Reliability
42,2019-12-31 10:30:00+00:00,131.1590,94.5056,78.4712,16.0344,210.5552,94.5056,78.4712,16.0344,210.5552,1.0
43,2019-12-31 10:45:00+00:00,134.5034,97.3787,81.1246,16.2542,212.2648,97.3787,81.1246,16.2542,212.2648,1.0
44,2019-12-31 11:00:00+00:00,136.8639,99.4104,82.9942,16.4162,213.4131,99.4104,82.9942,16.4162,213.4131,1.0
45,2019-12-31 11:15:00+00:00,138.2305,100.5875,84.0652,16.5223,214.0309,100.5875,84.0652,16.5223,214.0309,1.0
46,2019-12-31 11:30:00+00:00,138.5973,100.9022,84.3290,16.5732,214.1346,100.9022,84.3290,16.5732,214.1346,1.0
47,2019-12-31 11:45:00+00:00,137.9626,100.3524,83.7834,16.5689,213.7277,100.3524,83.7834,16.5689,213.7277,1.0
48,2019-12-31 12:00:00+00:00,136.3294,98.9590,82.4766,16.4824,212.9138,98.9590,82.4766,16.4824,212.9138,1.0
49,2019-12-31 12:15:00+00:00,133.7044,96.7305,80.4179,16.3126,211.6732,96.7305,80.4179,16.3126,211.6732,1.0
50,2019-12-31 12:30:00+00:00,130.0990,93.6631,77.5794,16.0837,209.8583,93.6631,77.5794,16.0837,209.8583,1.0


,# Observation period,TOA,Clear sky GHI,Clear sky BHI,Clear sky DHI,Clear sky BNI,GHI,BHI,DHI,BNI,Reliability
42,2019-12-31 10:30:00+00:00,131.4660,94.2978,78.3354,15.9624,209.6990,92.1881,74.9243,17.2639,200.5964,1.0
43,2019-12-31 10:45:00+00:00,135.0570,97.3852,81.2028,16.1824,211.5981,93.5265,74.6596,18.8669,194.5532,1.0
44,2019-12-31 11:00:00+00:00,137.6635,99.6347,83.2890,16.3457,212.9268,98.4517,81.2562,17.1955,207.7092,1.0
45,2019-12-31 11:15:00+00:00,139.2744,101.0313,84.5770,16.4543,213.7200,101.0313,84.5770,16.4543,213.7200,1.0
46,2019-12-31 11:30:00+00:00,139.8826,101.5661,85.0566,16.5094,213.9978,101.5661,85.0566,16.5094,213.9978,1.0
47,2019-12-31 11:45:00+00:00,139.4857,101.2351,84.7238,16.5114,213.7667,101.2351,84.7238,16.5114,213.7667,1.0
48,2019-12-31 12:00:00+00:00,138.0853,100.0559,83.6305,16.4255,213.1476,100.0559,83.6305,16.4255,213.1476,1.0
49,2019-12-31 12:15:00+00:00,135.6874,98.0357,81.7845,16.2512,212.1250,98.0357,81.7845,16.2512,212.1250,1.0
50,2019-12-31 12:30:00+00:00,132.3022,95.1715,79.1505,16.0210,210.5436,95.1715,79.1505,16.0210,210.5436,1.0


,# Observation period,TOA,Clear sky GHI,Clear sky BHI,Clear sky DHI,Clear sky BNI,GHI,BHI,DHI,BNI,Reliability
42,2019-12-31 10:30:00+00:00,128.7457,92.4755,77.1252,15.3503,210.8207,92.4755,77.1252,15.3503,210.8207,1.0
43,2019-12-31 10:45:00+00:00,132.5785,95.7792,80.2260,15.5532,212.9601,95.7792,80.2260,15.5532,212.9601,1.0
44,2019-12-31 11:00:00+00:00,135.4351,98.2533,82.5536,15.6996,214.5187,98.2533,82.5536,15.6996,214.5187,1.0
45,2019-12-31 11:15:00+00:00,137.3035,99.8815,84.0892,15.7923,215.5371,99.8815,84.0892,15.7923,215.5371,1.0
46,2019-12-31 11:30:00+00:00,138.1755,100.6534,84.8206,15.8329,216.0403,100.6534,84.8206,15.8329,216.0403,1.0
47,2019-12-31 11:45:00+00:00,138.0475,100.5639,84.7418,15.8221,216.0397,100.2795,84.2420,16.0375,214.7630,1.0
48,2019-12-31 12:00:00+00:00,136.9200,99.6067,83.8463,15.7604,215.5164,94.1017,74.1831,19.9186,190.6517,1.0
49,2019-12-31 12:15:00+00:00,134.7978,97.7878,82.1410,15.6468,214.4559,90.4277,69.2881,21.1396,180.9015,1.0
50,2019-12-31 12:30:00+00:00,131.6900,95.1254,79.6464,15.4790,212.8480,89.4275,69.7941,19.6333,186.5382,1.0


>>>A ce stade, nous avons des données athmosphériques avec une résolution temporelle de **15 minutes**.
>>>
>>>Cependant, le jeu de données dans lequel nous souhaitons ajouter ces données athmosphériques a une résolution temporelle de **30 minutes**.
>>>
>>>Or les données que nous avons collectées concernant l'athmosphères sont toutes cumulatives (à l'exception de la fiabilité `Reliability`) : pour avoir les données pour 30 minutes nous devons maintenant sommer les lignes deux à deux.
>>>
>>>Pour que la variable `Reliability` reste pertinente, on sa remplace par la moyenne entre le temp *t* et le temp *t-1*.

In [71]:
# On parcours l'ensemble des datasets
for df in all_cams.values():
  # Les colonnes à sommer vont de la deuxième à l'avant dernière (index 1 à -1)

  # On sélectionne les heures +15min et +45min (toutes les lignes impaires)
  df_impair = df.iloc[1:-1:2, 1:]

  # On sélectionne les heures +0min et +30min (toutes les lignes paires)
  # La première ligne n'est pas traitée et importe peu car a lieu en pleine nuit quand il n'y a pas de soleil
  df_pair = df.iloc[2::2, 1:]

  # On modifie l'index de df_impair pour pouvoir le sommer à df_pair
  df_impair.set_index(df_pair.index, inplace=True)

  # On met à jour le dataset avec les valeurs sommées
  df.iloc[2::2, 1:] = df_pair.add(df_impair)

  # On divise par deux la variable 'Reliability' (dernière colonne) pour avoir sa moyenne
  df.iloc[2::2, -1] /= 2

  # On affiche un échantillon
  display(df.iloc[42:51, :])
  print(df.shape)

,# Observation period,TOA,Clear sky GHI,Clear sky BHI,Clear sky DHI,Clear sky BNI,GHI,BHI,DHI,BNI,Reliability
42,2019-12-31 10:30:00+00:00,251.3930,184.0368,155.8095,28.2272,436.1868,184.0368,155.8095,28.2272,436.1868,1.0
43,2019-12-31 10:45:00+00:00,131.5526,97.1360,82.7308,14.4052,221.3223,97.1360,82.7308,14.4052,221.3223,1.0
44,2019-12-31 11:00:00+00:00,265.7156,196.5724,167.6386,28.9338,444.0508,196.5724,167.6386,28.9338,444.0508,1.0
45,2019-12-31 11:15:00+00:00,135.7882,100.8832,86.2814,14.6018,223.6242,100.8832,86.2814,14.6018,223.6242,1.0
46,2019-12-31 11:30:00+00:00,272.2093,202.3502,173.1224,29.2278,447.6552,202.3502,173.1224,29.2278,447.6552,1.0
47,2019-12-31 11:45:00+00:00,136.0591,101.1840,86.5821,14.6019,223.9572,101.1840,86.5821,14.6019,223.9572,1.0
48,2019-12-31 12:00:00+00:00,270.7628,201.2153,172.0872,29.1282,447.3532,201.2153,172.0872,29.1282,447.3532,1.0
49,2019-12-31 12:15:00+00:00,132.3607,98.0161,83.6177,14.3984,222.3308,98.0161,83.6177,14.3984,222.3308,1.0
50,2019-12-31 12:30:00+00:00,261.4009,193.1710,164.5533,28.6177,443.0655,193.1710,164.5533,28.6177,443.0655,1.0


(175488, 11)


,# Observation period,TOA,Clear sky GHI,Clear sky BHI,Clear sky DHI,Clear sky BNI,GHI,BHI,DHI,BNI,Reliability
42,2019-12-31 10:30:00+00:00,247.5166,183.3908,157.4185,25.9723,447.5976,183.3908,157.4185,25.9723,447.5976,1.0
43,2019-12-31 10:45:00+00:00,129.4494,96.6949,83.4551,13.2398,226.8875,96.6949,83.4551,13.2398,226.8875,1.0
44,2019-12-31 11:00:00+00:00,261.4069,195.6120,169.0276,26.5844,455.1118,195.6120,169.0276,26.5844,455.1118,1.0
45,2019-12-31 11:15:00+00:00,133.4866,100.2871,86.8832,13.4038,229.0667,100.2871,86.8832,13.4038,229.0667,1.0
46,2019-12-31 11:30:00+00:00,267.5169,201.0830,174.2603,26.8225,458.5017,201.0830,174.2603,26.8225,458.5017,1.0
47,2019-12-31 11:45:00+00:00,133.5861,100.4401,87.0503,13.3898,229.3367,100.4401,87.0503,13.3898,229.3367,1.0
48,2019-12-31 12:00:00+00:00,265.7421,199.6559,172.9445,26.7114,458.0754,199.6559,172.9445,26.7114,458.0754,1.0
49,2019-12-31 12:15:00+00:00,129.7461,97.1301,83.9170,13.2131,227.6228,97.1301,83.9170,13.2131,227.6228,1.0
50,2019-12-31 12:30:00+00:00,256.1128,191.3335,165.0624,26.2711,453.6114,191.3335,165.0624,26.2711,453.6114,1.0


(175488, 11)


,# Observation period,TOA,Clear sky GHI,Clear sky BHI,Clear sky DHI,Clear sky BNI,GHI,BHI,DHI,BNI,Reliability
42,2019-12-31 10:30:00+00:00,258.0040,185.3157,153.5272,31.7885,418.7921,185.3157,153.5272,31.7885,418.7921,1.0
43,2019-12-31 10:45:00+00:00,134.5034,97.3787,81.1246,16.2542,212.2648,97.3787,81.1246,16.2542,212.2648,1.0
44,2019-12-31 11:00:00+00:00,271.3673,196.7891,164.1188,32.6704,425.6779,196.7891,164.1188,32.6704,425.6779,1.0
45,2019-12-31 11:15:00+00:00,138.2305,100.5875,84.0652,16.5223,214.0309,100.5875,84.0652,16.5223,214.0309,1.0
46,2019-12-31 11:30:00+00:00,276.8278,201.4897,168.3942,33.0955,428.1655,201.4897,168.3942,33.0955,428.1655,1.0
47,2019-12-31 11:45:00+00:00,137.9626,100.3524,83.7834,16.5689,213.7277,100.3524,83.7834,16.5689,213.7277,1.0
48,2019-12-31 12:00:00+00:00,274.2920,199.3114,166.2600,33.0513,426.6415,199.3114,166.2600,33.0513,426.6415,1.0
49,2019-12-31 12:15:00+00:00,133.7044,96.7305,80.4179,16.3126,211.6732,96.7305,80.4179,16.3126,211.6732,1.0
50,2019-12-31 12:30:00+00:00,263.8034,190.3936,157.9973,32.3963,421.5315,190.3936,157.9973,32.3963,421.5315,1.0


(175488, 11)


,# Observation period,TOA,Clear sky GHI,Clear sky BHI,Clear sky DHI,Clear sky BNI,GHI,BHI,DHI,BNI,Reliability
42,2019-12-31 10:30:00+00:00,258.3717,184.6907,153.0459,31.6448,416.8769,182.4777,149.4691,33.0087,407.3221,1.0
43,2019-12-31 10:45:00+00:00,135.0570,97.3852,81.2028,16.1824,211.5981,93.5265,74.6596,18.8669,194.5532,1.0
44,2019-12-31 11:00:00+00:00,272.7205,197.0199,164.4918,32.5281,424.5249,191.9782,155.9158,36.0624,402.2624,1.0
45,2019-12-31 11:15:00+00:00,139.2744,101.0313,84.5770,16.4543,213.7200,101.0313,84.5770,16.4543,213.7200,1.0
46,2019-12-31 11:30:00+00:00,279.1570,202.5974,169.6336,32.9637,427.7178,202.5974,169.6336,32.9637,427.7178,1.0
47,2019-12-31 11:45:00+00:00,139.4857,101.2351,84.7238,16.5114,213.7667,101.2351,84.7238,16.5114,213.7667,1.0
48,2019-12-31 12:00:00+00:00,277.5710,201.2910,168.3543,32.9369,426.9143,201.2910,168.3543,32.9369,426.9143,1.0
49,2019-12-31 12:15:00+00:00,135.6874,98.0357,81.7845,16.2512,212.1250,98.0357,81.7845,16.2512,212.1250,1.0
50,2019-12-31 12:30:00+00:00,267.9896,193.2072,160.9350,32.2722,422.6686,193.2072,160.9350,32.2722,422.6686,1.0


(175488, 11)


,# Observation period,TOA,Clear sky GHI,Clear sky BHI,Clear sky DHI,Clear sky BNI,GHI,BHI,DHI,BNI,Reliability
42,2019-12-31 10:30:00+00:00,252.6989,180.8398,150.4024,30.4374,418.8627,180.8398,150.4024,30.4374,418.8627,1.0
43,2019-12-31 10:45:00+00:00,132.5785,95.7792,80.2260,15.5532,212.9601,95.7792,80.2260,15.5532,212.9601,1.0
44,2019-12-31 11:00:00+00:00,268.0136,194.0325,162.7796,31.2528,427.4788,194.0325,162.7796,31.2528,427.4788,1.0
45,2019-12-31 11:15:00+00:00,137.3035,99.8815,84.0892,15.7923,215.5371,99.8815,84.0892,15.7923,215.5371,1.0
46,2019-12-31 11:30:00+00:00,275.4790,200.5349,168.9098,31.6252,431.5774,200.5349,168.9098,31.6252,431.5774,1.0
47,2019-12-31 11:45:00+00:00,138.0475,100.5639,84.7418,15.8221,216.0397,100.2795,84.2420,16.0375,214.7630,1.0
48,2019-12-31 12:00:00+00:00,274.9675,200.1706,168.5881,31.5825,431.5561,194.3812,158.4251,35.9561,405.4147,1.0
49,2019-12-31 12:15:00+00:00,134.7978,97.7878,82.1410,15.6468,214.4559,90.4277,69.2881,21.1396,180.9015,1.0
50,2019-12-31 12:30:00+00:00,266.4878,192.9132,161.7874,31.1258,427.3039,179.8552,139.0822,40.7729,367.4397,1.0


(175488, 11)


>>>On renomme les variables des datasets (à l'exeption de la variable `datetime_utc` qui servira à la fusion) en les faisants débuter par le sigle du point d'intérêt correspondant :

In [73]:
for ville, df in all_cams.items() :
    mon_dico = {'# Observation period' : 'datetime_utc',
                'TOA' : ville + '_TOA',
                'Clear sky GHI' : ville + '_Clear sky GHI',
                'Clear sky BHI' : ville + '_Clear sky BHI',
                'Clear sky DHI' : ville + '_Clear sky DHI',
                'Clear sky BNI' : ville + '_Clear sky BNI',
                'GHI' : ville + '_GHI',
                'BHI' : ville + '_BHI',
                'DHI' : ville + '_DHI',
                'BNI' : ville + '_BNI',
                'Reliability' : ville + '_Reliability'}
    df.rename(mon_dico, axis=1, inplace=True)

for df in all_cams.values():
  display(df.head())

,datetime_utc,CRU_TOA,CRU_Clear sky GHI,CRU_Clear sky BHI,CRU_Clear sky DHI,CRU_Clear sky BNI,CRU_GHI,CRU_BHI,CRU_DHI,CRU_BNI,CRU_Reliability
0,2019-12-31 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2019-12-31 00:15:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,2019-12-31 00:30:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,2019-12-31 00:45:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,2019-12-31 01:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


,datetime_utc,SEL_TOA,SEL_Clear sky GHI,SEL_Clear sky BHI,SEL_Clear sky DHI,SEL_Clear sky BNI,SEL_GHI,SEL_BHI,SEL_DHI,SEL_BNI,SEL_Reliability
0,2019-12-31 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2019-12-31 00:15:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,2019-12-31 00:30:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,2019-12-31 00:45:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,2019-12-31 01:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


,datetime_utc,SVT_TOA,SVT_Clear sky GHI,SVT_Clear sky BHI,SVT_Clear sky DHI,SVT_Clear sky BNI,SVT_GHI,SVT_BHI,SVT_DHI,SVT_BNI,SVT_Reliability
0,2019-12-31 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2019-12-31 00:15:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,2019-12-31 00:30:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,2019-12-31 00:45:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,2019-12-31 01:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


,datetime_utc,BRA_TOA,BRA_Clear sky GHI,BRA_Clear sky BHI,BRA_Clear sky DHI,BRA_Clear sky BNI,BRA_GHI,BRA_BHI,BRA_DHI,BRA_BNI,BRA_Reliability
0,2019-12-31 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2019-12-31 00:15:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,2019-12-31 00:30:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,2019-12-31 00:45:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,2019-12-31 01:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


,datetime_utc,EYG_TOA,EYG_Clear sky GHI,EYG_Clear sky BHI,EYG_Clear sky DHI,EYG_Clear sky BNI,EYG_GHI,EYG_BHI,EYG_DHI,EYG_BNI,EYG_Reliability
0,2019-12-31 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2019-12-31 00:15:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,2019-12-31 00:30:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,2019-12-31 00:45:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,2019-12-31 01:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


>>>On fusionne les datasets en se basant sur la variable `datetime_utc` qui contient la date et l'heure des observations :  

In [74]:
for df in all_cams.values():
  df_astro = df_astro.merge(right=df, on='datetime_utc', how='left')

>>>On affiche le début du jeu de données pour avoir une première visualisation des nouvelles données :

In [78]:
print(df_astro.shape)
display(df_astro.iloc[23:28, :])

(87696, 70)


,datetime_utc,Périmètre,Nature,Consommation,Solaire,Ech. physiques,Stockage batterie,Déstockage batterie,TCO Solaire (%),TCH Solaire (%),...,EYG_TOA,EYG_Clear sky GHI,EYG_Clear sky BHI,EYG_Clear sky DHI,EYG_Clear sky BNI,EYG_GHI,EYG_BHI,EYG_DHI,EYG_BNI,EYG_Reliability
23,2020-01-01 10:30:00+00:00,PACA,Données définitives,5355.0,587.0,3248.0,-,-,10.96,43.74,...,253.2484,181.5259,150.7824,30.7435,419.0294,181.0210,149.8874,31.1336,416.6206,1.0
24,2020-01-01 11:00:00+00:00,PACA,Données définitives,5389.0,637.0,3222.0,-,-,11.82,47.47,...,268.6971,194.6732,162.9588,31.7144,426.8785,173.7657,126.2110,47.5547,330.7245,1.0
25,2020-01-01 11:30:00+00:00,PACA,Données définitives,5510.0,665.0,3325.0,-,-,12.07,49.55,...,276.2950,201.1038,168.8604,32.2434,430.1928,176.9042,126.1173,50.7869,321.3643,1.0
26,2020-01-01 12:00:00+00:00,PACA,Données définitives,5396.0,667.0,3208.0,-,-,12.36,49.70,...,275.9122,200.6609,168.3135,32.3474,429.3926,177.5202,127.8147,49.7055,326.1265,1.0
27,2020-01-01 12:30:00+00:00,PACA,Données définitives,5453.0,650.0,3289.0,-,-,11.92,48.44,...,267.5549,193.3875,161.3886,31.9990,424.5634,173.1829,126.9076,46.2753,333.8453,1.0


>>### E - Enregistrement des données CAMS collectées ###
>>>
>>>Nous avons terminé la collecte des données explicatives relatives à l'`athmosphère` pour chaque point significatif du point de vue de la production d'énergie solaire, via le jeu de données **CAMS solar radiation time-serie**.
>>>
>>>Nous enregistrons notre jeu de données actuel pour clore la troisième partie de notre travail de collecte.

In [79]:
# On enregistre ce dataset contenant les données de production, astronomiques et météo partielle avant ajout d'autres variables
df_astro.to_csv(chemin_datasets_meteo + "cams_2020_2024.csv", index=False)

In [80]:
# Exemple de la manière dont charger ce dataset production+astro final :
df_cams = pd.read_csv(chemin_datasets_meteo + "cams_2020_2024.csv", parse_dates=['datetime_utc'])
display(df_cams.head())
df_cams.dtypes

/tmp/ipython-input-3359716260.py:2: DtypeWarning: Columns (6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_cams = pd.read_csv(chemin_datasets_meteo + "cams_2020_2024.csv", parse_dates=['datetime_utc'])


,datetime_utc,Périmètre,Nature,Consommation,Solaire,Ech. physiques,Stockage batterie,Déstockage batterie,TCO Solaire (%),TCH Solaire (%),...,EYG_TOA,EYG_Clear sky GHI,EYG_Clear sky BHI,EYG_Clear sky DHI,EYG_Clear sky BNI,EYG_GHI,EYG_BHI,EYG_DHI,EYG_BNI,EYG_Reliability
0,2019-12-31 23:00:00+00:00,PACA,Données définitives,6123.0,0.0,3332.0,-,-,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2019-12-31 23:30:00+00:00,PACA,Données définitives,5907.0,0.0,2837.0,-,-,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,2020-01-01 00:00:00+00:00,PACA,Données définitives,5724.0,0.0,2668.0,-,-,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,2020-01-01 00:30:00+00:00,PACA,Données définitives,5749.0,0.0,2754.0,-,-,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,2020-01-01 01:00:00+00:00,PACA,Données définitives,5700.0,0.0,2886.0,-,-,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


,0
datetime_utc,"datetime64[ns, UTC]"
Périmètre,object
Nature,object
Consommation,float64
Solaire,float64
...,...
EYG_GHI,float64
EYG_BHI,float64
EYG_DHI,float64
EYG_BNI,float64
